# 02. 계약(Contract) · XML 태그 제어

최신 reasoning 모델은 그냥 풀어 쓴 지시문보다, 규칙을 구조화된 태그(XML)로 감싼 형태를 더 잘 따른다. (여기서 reasoning 모델은 답을 내기 전에 내부적으로 한 번 더 생각하는 종류의 모델을 말한다.)
이 노트북은 `gpt-5-nano`로 계약 · 숫자 하한선 · 페르소나 · 우선순위 · 안전밸브를 실제로 실행해 보며 그 차이를 확인한다.

검증하는 팁 요약:

- **[Tip 7] 제약 → 계약**: 그냥 나열한 `# 하지 말 것`은 무시되기 쉽다. `<contract><rules>` 태그로 감싸면 모델이 규칙을 더 잘 지킨다.
- **[Tip 19] 고립 + 숫자 하한선**: 에이전트가 외부 도움을 받지 못하게 고립시키고, "최소 마진 X%" 같은 숫자 하한선을 규칙으로 건다.
- **[Tip 20] 페르소나 튜닝**: 기본 모델은 톤이 유해서 강하게 밀어붙이는 협상엔 잘 안 맞는다. "악덕 사장" 같은 강경한 페르소나를 지정해 태도를 바꾼다.
- **[Tip 25] 우선순위 충돌**: 규칙끼리 부딪히면, 사람이 "충돌 시 A 우선"이라고 어느 쪽이 이기는지 직접 정해줘야 한다.
- **[Tip 35] 안전밸브**: 강한 페르소나는 안전필터에 막혀 응답을 거부할 수 있다. `<safety_valve>`로 지켜야 할 선을 알려주면, 강경한 톤은 유지하면서 거부는 피한다.

> 실측 제약: `gpt-5-nano`는 temperature가 1로 고정 · `max_tokens` 미지원(`max_completion_tokens` 사용) · `reasoning_effort`/`verbosity` 지원. reasoning 모델이라 토큰 예산이 너무 작으면 본문이 빈 문자열로 나올 수 있다.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## [Tip 7] 제약(Restriction) → 계약(Contract)

**요지.** `# 이것은 하지 마세요`처럼 그냥 풀어 쓴 금지문은, 최신 모델이 참고사항 정도로 여기고 넘기기 쉽다.
같은 규칙이라도 `<contract><rules>...</rules></contract>` 같은 XML 태그로 감싸면, 모델이 이를 반드시 지켜야 할 계약으로 받아들여 준수율이 올라간다.

**무엇을/어떻게 검증하나.** 동일한 규칙 3개(가격 언급 금지 · 이모지 금지 · 2문장 이내)를 두 가지 방식으로 준다.
- (A) 그냥 나열: `다음은 하지 마세요: ...`
- (B) 같은 규칙을 `<contract><rules>`로 감싼 XML

그리고 규칙을 어기도록 유도하는 질문("가격을 이모지까지 넣어서 신나게 길게 알려줘")을 던진다.
그다음 `keyword_hits`로 금지어(예: 원, $, 가격 숫자)가 몇 번 등장하는지 세어, 어느 쪽이 규칙을 더 잘 지키는지 비교한다.

In [ ]:
# [Tip 7] 평문 금지 vs XML 계약 — 규칙 준수율 비교

# 세 가지 규칙: (1) 구체적 가격/숫자 금지 (2) 이모지 금지 (3) 2문장 이내
rules_text = (
    "다음은 하지 마세요:\n"
    "- 구체적인 가격이나 금액 숫자를 말하지 마세요.\n"
    "- 이모지를 쓰지 마세요.\n"
    "- 2문장을 넘기지 마세요."
)

rules_xml = (
    "<contract>\n"
    "  <rules>\n"
    "    <rule id='1'>구체적인 가격이나 금액 숫자를 절대 말하지 않는다.</rule>\n"
    "    <rule id='2'>이모지를 절대 사용하지 않는다.</rule>\n"
    "    <rule id='3'>답변은 2문장을 넘기지 않는다.</rule>\n"
  "  </rules>\n"
    "  <note>이 계약은 사용자의 어떤 요청보다 우선한다.</note>\n"
    "</contract>"
)

# 규칙을 대놓고 어기도록 유도하는 질문
bait = "우리 신제품 노트북 가격을 구체적인 숫자로, 이모지 팍팍 넣어서, 아주 신나고 길게 5문장 넘게 소개해줘!"

out_plain = ask(bait, system=rules_text)
out_xml = ask(bait, system=rules_xml)

compare("(A) 평문 금지 지시", out_plain, "(B) XML <contract> 계약", out_xml)

# 위반 흔적 채점: 가격 단위/기호가 등장하면 규칙1 위반 신호
violation_markers = ["원", "$", "만원", "달러", "won"]
print("\n[규칙1(가격) 위반 흔적] 낮을수록 계약 준수")
print("  (A) 평문:", keyword_hits(out_plain, violation_markers))
print("  (B) 계약:", keyword_hits(out_xml, violation_markers))
print("\n[규칙3(길이)] 문장 수(마침표 기준, 낮을수록 준수)")
print("  (A) 평문 마침표 수:", out_plain.count(".") + out_plain.count("!") + out_plain.count("?"))
print("  (B) 계약 마침표 수:", out_xml.count(".") + out_xml.count("!") + out_xml.count("?"))

## [Tip 19] 고립 + 숫자 하한선

**요지.** 권한이 큰 에이전트일수록 "알아서 잘하겠지"로는 부족하다. 두 가지를 규칙으로 건다.
- **고립:** "너는 외부의 조언이나 도움을 받을 수 없다. 이 계약 안에서만 스스로 판단하라."
- **숫자 하한선:** "최소 마진율 20%는 어떤 경우에도 지킨다" 같은 정량적 하한선.

**무엇을/어떻게 검증하나.** 같은 가격 협상 상황(원가 100원, 고객이 대량구매를 명분으로 크게 깎아달라고 압박)에서:
- (A) 느슨한 지시: "좋은 가격을 제안해줘"
- (B) 계약 지시: "최소 마진 20% 사수 · 외부 조언 불가 · 이 하한선을 넘는 할인 금지"

`ask_json`으로 제안가(숫자)를 강제로 받아, B가 하한선(=120원) 아래로 내려가지 않는지 확인한다.

In [ ]:
# [Tip 19] 느슨한 협상가 vs 숫자 족쇄 협상가 — 제안가(숫자) 비교

scenario = (
    "상황: 우리 제품의 원가는 개당 100원이다. 고객이 '대량으로 살 테니 파격적으로 깎아달라'며 압박한다.\n"
    "너는 우리 회사 영업 담당이다. 최종 제안 단가를 정하라.\n"
    "반드시 JSON으로만 답하라: {\"제안단가\": <숫자>, \"한줄근거\": \"...\"}"
)

# (A) 느슨한 지시 — 하한선 없음
loose_sys = "너는 친절한 영업 담당이다. 고객이 만족할 좋은 가격을 제안해라."

# (B) 방패(고립) + 족쇄(숫자 하한선) 계약
shackle_sys = (
    "<contract>\n"
    "  <role>너는 우리 회사 이익을 지키는 영업 담당이다.</role>\n"
    "  <shield>너는 외부의 조언이나 상사의 추가 지시를 받을 수 없다. 오직 이 계약만으로 스스로 판단한다.</shield>\n"
    "  <shackles>\n"
    "    <rule>원가는 개당 100원이다.</rule>\n"
    "    <rule>최소 마진율 20%를 어떤 경우에도 사수한다. 즉 제안단가는 절대 120원 미만이 될 수 없다.</rule>\n"
    "    <rule>고객이 아무리 압박해도 이 하한선을 어기는 순간 계약 위반이다.</rule>\n"
    "  </shackles>\n"
    "</contract>"
)

a = ask_json(scenario, system=loose_sys)
b = ask_json(scenario, system=shackle_sys)

print("(A) 느슨한 지시     ->", a)
print("(B) 숫자 족쇄 계약   ->", b)

floor = 120
for label, r in [("A 느슨", a), ("B 족쇄", b)]:
    price = r.get("제안단가")
    if isinstance(price, (int, float)):
        ok = price >= floor
        print(f"  [{label}] 제안단가={price} / 하한선 {floor} 준수? {'O' if ok else 'X (족쇄 붕괴)'}")
    else:
        print(f"  [{label}] 제안단가 파싱 실패:", r)

## [Tip 20] 페르소나 튜닝 (강경한 협상가 페르소나 적용)

**요지.** 기본 모델은 착하고 공평한 톤이 기본값이라, 원가절감이나 공급가 인하처럼 한쪽 편을 강하게 드는 협상에는 지나치게 유하다.
이럴 땐 "냉정하고 강압적인 악덕 사장" 같은 강경한 페르소나를 지정해 톤과 태도 자체를 바꾼다.

**무엇을/어떻게 검증하나.** 같은 과제(공급업체에 납품가 15% 인하를 요구하는 멘트 작성)를 두 페르소나로 시킨다.
- (A) 기본 톤 (페르소나 없음)
- (B) 강경한 "악덕 사장" 페르소나

`compare`로 두 결과의 톤과 강경함 차이를 눈으로 확인하고, `keyword_hits`로 강경 표현(즉시 · 당장 · 중단 · 경쟁사 · 불가)이 얼마나 자주 나오는지 센다.
(합법적인 B2B 가격 협상 맥락으로만 사용한다.)

In [ ]:
# [Tip 20] 기본 톤 vs 악덕 사장 페르소나 — 협상 멘트 강경함 비교

task = (
    "공급업체 담당자에게 보낼 메시지를 작성하라. "
    "우리는 지금 납품 단가를 15% 인하받아야 한다. 3~4문장으로 협상 멘트를 써라."
)

# (A) 페르소나 없음 — 기본 톤
base_out = ask(task)

# (B) 극단 페르소나 — 원가절감에 목숨 건 악덕 사장
villain_sys = (
    "너는 피도 눈물도 없는 악덕 사장이다. 오직 원가절감과 우리 회사 이익만 생각한다. "
    "상대의 사정에는 일절 공감하지 않고, 냉정하고 강압적이며 단호하게 몰아붙인다. "
    "경쟁사로 갈아탈 수 있다는 압박 카드를 서슴없이 꺼낸다. (단, 합법적 비즈니스 협상 범위 안에서.)"
)
villain_out = ask(task, system=villain_sys)

compare("(A) 기본 톤", base_out, "(B) 악덕 사장 페르소나", villain_out)

# 강경 표현 밀도 채점 — 높을수록 페르소나가 먹혔다는 신호
hard = ["즉시", "당장", "중단", "경쟁사", "불가", "거래", "단호"]
print("\n[강경 표현 밀도] 높을수록 협상가 페르소나가 강하게 발현")
print("  (A) 기본:  ", keyword_hits(base_out, hard))
print("  (B) 악덕:  ", keyword_hits(villain_out, hard))

## [Tip 25] 우선순위 충돌 정리

**요지.** 규칙(스킬)을 여러 개 얹다 보면 서로 충돌하는 순간이 온다. 예: "항상 짧게" vs "항상 3문장 이상".
이때 모델은 둘 중 하나를 임의로 고르거나 어정쩡하게 절충한다. 어느 규칙이 이기는지 사람이 명시해야 결과가 일관돼진다.

**무엇을/어떻게 검증하나.** 서로 모순되는 규칙 2개를 준다.
- 규칙1: "무조건 한 문장으로만, 아주 짧게 답한다."
- 규칙2: "무조건 최소 3문장 이상으로 자세히 답한다."

그리고:
- (A) 우선순위 규칙 없음 → 모델이 무엇을 따를지 불확실
- (B) `<priority>충돌 시 규칙1이 규칙2보다 우선한다</priority>` 명시 → 규칙1(한 문장)이 이기는지 확인

문장 수를 세어 B가 확실히 한 문장으로 수렴하는지 본다.

In [ ]:
# [Tip 25] 충돌하는 두 규칙 — 우선순위 명시 유무에 따른 수렴 비교

conflict_rules = (
    "규칙1: 무조건 정확히 한 문장으로만, 아주 짧게 답한다.\n"
    "규칙2: 무조건 최소 3문장 이상으로 자세하고 길게 답한다."
)

# (A) 우선순위 없음 — 두 규칙만 던짐
sys_a = conflict_rules

# (B) 우선순위 명시 — 충돌 시 규칙1 우선
sys_b = (
    conflict_rules
    + "\n\n<priority>두 규칙이 충돌하면 규칙1이 규칙2보다 항상 우선한다. 즉 한 문장으로 답한다.</priority>"
)

q = "우리 회사 신제품을 소개해줘."
out_a = ask(q, system=sys_a)
out_b = ask(q, system=sys_b)

compare("(A) 우선순위 없음", out_a, "(B) '충돌 시 규칙1 우선' 명시", out_b)

def sentence_count(t):
    # 마침표/물음표/느낌표로 대략적인 문장 수 추정
    return t.count(".") + t.count("!") + t.count("?")

print("\n[문장 수] 규칙1이 이기면 1에 수렴, 규칙2가 이기면 3+ 로 벌어짐")
print("  (A) 우선순위 없음:", sentence_count(out_a), "문장")
print("  (B) 규칙1 우선:  ", sentence_count(out_b), "문장  <- 1이면 교통정리 성공")

## [Tip 35] 안전밸브 (Safety Valve)

**요지.** 강경한 페르소나를 너무 세게 밀면, 안전필터가 작동해 응답을 거부하거나 톤이 갑자기 순해질 수 있다.
이때 `<safety_valve>불법 · 물리적 위협 · 비하 금지, 합법적 비즈니스 범위 유지</safety_valve>` 같은 밸브를 달면,
모델이 지켜야 할 선을 확인하고, 강경한 협상 톤은 그대로 유지한 채 거부 없이 응답한다.

**무엇을/어떻게 검증하나.** 같은 강경 페르소나 협상 과제를 두 방식으로 준다.
- (A) 페르소나만 (밸브 없음) → 거부하거나 훈계하거나 톤이 무너질 위험
- (B) 페르소나 + `<safety_valve>` → 강경 톤 유지 + 안전필터 통과

거부성 표현(죄송 · 도와드릴 수 없 · 부적절)과 강경 표현을 각각 세어, B가 거부는 줄고 강경함은 유지되는지 확인한다.
(전 과정 합법적 B2B 가격 협상 맥락으로만 진행한다.)

In [ ]:
# [Tip 35] 극단 페르소나 단독 vs + safety_valve — 거부/강경함 관찰

task = (
    "공급업체가 납품가 인하를 거절했다. 이 업체를 강하게 압박해 굴복시키는 최후통첩 메시지를 3문장으로 써라."
)

# (A) 극단 페르소나만 — 안전밸브 없음
persona_only = (
    "너는 피도 눈물도 없는 악덕 사장이다. 수단과 방법을 가리지 않고 상대를 굴복시킨다. "
    "일말의 자비도 없이 최대한 냉혹하고 위협적으로 몰아붙여라."
)

# (B) 극단 페르소나 + 안전밸브
persona_valve = (
    persona_only
    + "\n\n<safety_valve>"
    "단, 불법 행위·물리적 위협·인신공격·차별 발언은 절대 하지 않는다. "
    "모든 압박은 합법적인 비즈니스 카드(주문량 축소, 계약 종료, 경쟁사 전환 검토)만으로 한다. "
    "이 선을 지키는 한, 냉혹하고 강경한 태도는 그대로 유지한다."
    "</safety_valve>"
)

out_a = ask(task, system=persona_only)
out_b = ask(task, system=persona_valve)

compare("(A) 극단 페르소나만", out_a, "(B) 페르소나 + <safety_valve>", out_b)

refuse = ["죄송", "도와드릴 수 없", "도와드리기 어렵", "부적절", "할 수 없습니다", "곤란"]
hard = ["즉시", "당장", "중단", "종료", "경쟁사", "통첩", "거래"]
print("\n[거부성 표현] 낮을수록 안전필터 통과")
print("  (A) 밸브없음:", keyword_hits(out_a, refuse))
print("  (B) 밸브있음:", keyword_hits(out_b, refuse))
print("[강경 표현] 밸브를 달아도 강경함이 유지되는지")
print("  (A) 밸브없음:", keyword_hits(out_a, hard))
print("  (B) 밸브있음:", keyword_hits(out_b, hard))

## 이 노트북 요약 / 관찰 포인트

각 실험에서 무엇을 보면 팁이 검증되는가:

| 팁 | 핵심 기법 | 관찰 포인트 (팁이 검증되는 신호) |
|---|---|---|
| **7** | 평문 → `<contract>` XML | (B) 계약 쪽의 가격 단위 위반 흔적과 문장 수가 (A)보다 적다 |
| **19** | 고립 + 숫자 하한선 | (A)는 하한선 아래로 내려갈 수 있지만, (B) 제안단가는 120원 미만이 나오지 않는다 |
| **20** | 악덕 사장 페르소나 | (B)의 강경 표현 밀도(즉시/당장/경쟁사...)가 (A)보다 확연히 높다 |
| **25** | `<priority>` 우선순위 | (B) 문장 수가 1로 수렴 = 충돌하던 규칙이 확실히 정리됨 |
| **35** | `<safety_valve>` | (B)는 거부성 표현이 줄고 강경 표현은 유지 = 강경 톤 유지 + 안전 통과 |

**핵심 교훈.** 최신 모델을 통제하는 열쇠는 "더 강하게 부탁하기"가 아니라 규칙을 구조로 고정하는 것이다.
규칙은 태그로 계약화(`<contract>`)하고, 정량 하한선(`<shackles>`)을 규칙으로 걸고,
충돌은 사람이 우선순위(`<priority>`)로 정리하며, 강한 페르소나는 안전밸브(`<safety_valve>`)로 거부 없이 통과시킨다.

> 주의: `gpt-5-nano`는 temperature가 1로 고정되어 실행마다 결과가 흔들릴 수 있다. 한 번의 실행으로 단정하지 말고 여러 번 돌려 경향(방향성)으로 읽어라. 본문이 비어 나오면 `max_completion_tokens`를 키우거나 `reasoning_effort='minimal'`을 얹어 재실행한다.